# Training YOLOv8s Multi-Class di Colab T4 GPU — Tahap 2
Human Safety Monitoring System — INaAI 2026.

**PENTING — aktifkan GPU dulu:** menu `Runtime` -> `Change runtime type` -> Hardware accelerator = **T4 GPU** -> Save.

Parameter training (seed, augmentasi) dibuat **identik** dengan `train.py` lokal agar hasil reproducible.

## 1. Verifikasi GPU aktif

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics==8.3.0
import ultralytics; ultralytics.checks()

## 3. Ambil dataset
Upload **`css-remapped.zip`** (156 MB) ke Google Drive Anda (taruh di root `My Drive`), lalu jalankan sel ini untuk mount + unzip.

Alternatif tanpa Drive: comment baris mount, uncomment `files.upload()` (lebih lambat & hilang jika sesi putus).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp '/content/drive/MyDrive/css-remapped.zip' /content/css-remapped.zip

# --- Alternatif upload langsung (jika tidak pakai Drive) ---
# from google.colab import files
# files.upload()   # pilih css-remapped.zip
# !mv css-remapped.zip /content/css-remapped.zip

!unzip -q -o /content/css-remapped.zip -d /content/
!ls /content/construction-site-safety

## 4. Tulis data.yaml (path absolut Colab, 7 kelas)

In [ ]:
data_yaml = '''path: /content/construction-site-safety
train: train/images
val:   valid/images
test:  test/images
names:
  0: Person
  1: Hardhat
  2: NO-Hardhat
  3: Safety-Vest
  4: NO-Safety-Vest
  5: Mask
  6: NO-Mask
'''
with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)
print(data_yaml)

## 5. Training (params identik dengan train.py lokal)
T4 ~ 30-60 menit untuk 50 epoch. Ultralytics otomatis pakai GPU.

In [ ]:
import random, numpy as np, torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

model = YOLO('yolov8s.pt')
model.train(
    data='/content/data.yaml',
    epochs=50, imgsz=640, batch=16, seed=SEED,
    close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    scale=0.5, translate=0.1, fliplr=0.5, flipud=0.0,
    project='runs', name='yolov8s_ppe',
    device=0,
)
print('Selesai. best.pt di runs/yolov8s_ppe/weights/best.pt')

## 6. Eval di test set (gerbang go/no-go: mAP@0.5 >= 0.50)

In [ ]:
best = YOLO('runs/yolov8s_ppe/weights/best.pt')
m = best.val(data='/content/data.yaml', split='test', iou=0.5)
print('mAP@0.5      :', round(float(m.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(m.box.map), 4))
print('precision    :', round(float(m.box.mp), 4))
print('recall       :', round(float(m.box.mr), 4))

## 7. Download hasil ke lokal
Unduh `best.pt` -> taruh di `weights/best.pt` repo. Juga unduh kurva training untuk report.

In [ ]:
from google.colab import files
import shutil
shutil.copy('runs/yolov8s_ppe/weights/best.pt', '/content/best.pt')
shutil.make_archive('/content/train_artifacts', 'zip', 'runs/yolov8s_ppe')
files.download('/content/best.pt')
files.download('/content/train_artifacts.zip')  # results.png, args.yaml, kurva loss/mAP

# (opsional) salin ke Drive juga:
# shutil.copy('/content/best.pt', '/content/drive/MyDrive/best.pt')